# Prepare Gold Labels

This notebook creates the canonical gold-label table used throughout the thesis pipeline. Gold labels represent human-reviewed category-manager judgments and must remain separate from weak labels generated by Snorkel. The goal is to produce one clean, validated, joinable file that can be merged into the feature-engineered contract dataset before Stage 1 and Stage 2 experiments.

The notebook also creates candidate samples for future labeling. Candidate sampling is only used to select contracts for human review; it does not create synthetic gold labels.

## 1. Environment Setup

We first load standard libraries, define project paths, and configure display settings. The notebook is intended to be run from the `notebooks/` directory, so all paths are defined relative to the project root.

In [68]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "Data" / "processed"
DATA_MANUAL = PROJECT_ROOT / "Data" / "manual"
REPORTS_DIR = PROJECT_ROOT / "reports" / "gold_labels"

DATA_MANUAL.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Manual data folder:", DATA_MANUAL)
print("Processed data folder:", DATA_PROCESSED)
print("Reports folder:", REPORTS_DIR)

Project root: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-
Manual data folder: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/manual
Processed data folder: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/processed
Reports folder: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/reports/gold_labels


## 2. Load the Contract Feature Dataset

The gold labels must be validated against the actual contract universe used in the modeling pipeline. This prevents category-manager labels from referring to contracts that were filtered out upstream or have incompatible identifiers.

In [69]:
FEATURE_DATA_PATH = DATA_PROCESSED / "contract_with_features_labeled.csv"

if not FEATURE_DATA_PATH.exists():
    FEATURE_DATA_PATH = DATA_PROCESSED / "contracts_with_features.csv"

if not FEATURE_DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find contract feature dataset. Expected either "
        "contract_with_features_labeled.csv or contracts_with_features.csv in Data/processed/."
    )

df_features = pd.read_csv(FEATURE_DATA_PATH, low_memory=False)

print("Loaded:", FEATURE_DATA_PATH.name)
print("Shape:", df_features.shape)
print("Unique contracts:", df_features["contract_id"].nunique() if "contract_id" in df_features else "contract_id missing")
print("Departments:", df_features["department"].nunique() if "department" in df_features else "department missing")

Loaded: contract_with_features_labeled.csv
Shape: (9201, 224)
Unique contracts: 2209
Departments: 16


## 3. Define or Load Manual Gold Labels

The preferred long-term workflow is to maintain manual labels in `Data/manual/gold_labels_manual.csv`. The fallback dictionary below is useful while the label set is still small and evolving. Each row should represent a human-reviewed contract-level label.

`gold_y = 1` means the contract should be considered a renegotiation candidate. `gold_y = 0` means the contract should not currently be prioritized for renegotiation.

In [70]:
MANUAL_LABEL_PATH = DATA_MANUAL / "gold_labels_manual.csv"

# Optional fallback while collecting labels.
# Replace or extend this dictionary when new category-manager labels arrive.
# Keep contract IDs as strings to avoid integer/float formatting issues.
gold_label_records = [
    # Bioprocessing
    {"contract_id": "504361", "contract_number": "504149", "gold_y": 1, "department": "Bioprocessing and Excipients"},
    {"contract_id": "613927", "contract_number": "612555", "gold_y": 1, "department": "Bioprocessing and Excipients"},
    {"contract_id": "345318", "contract_number": "346312", "gold_y": 0, "department": "Bioprocessing and Excipients"},
    {"contract_id": "1877",   "contract_number": "1877",   "gold_y": 0, "department": "Bioprocessing and Excipients"},
    
    # Global Strategic
    {"contract_id": "4821",   "contract_number": "4822",   "gold_y": 1, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "557",    "contract_number": "557",    "gold_y": 1, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "565",    "contract_number": "565",    "gold_y": 1, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "483149", "contract_number": "483178", "gold_y": 0, "department": "Global Strategic Outsourcing & Devices & Needles"},
    {"contract_id": "25777",  "contract_number": "25777",  "gold_y": 0, "department": "Global Strategic Outsourcing & Devices & Needles"},
    
    # Aseptic
    {"contract_id": "577893", "contract_number": "577893", "gold_y": 0, "department": "Aseptic Production"},
    {"contract_id": "370849", "contract_number": "370849", "gold_y": 0, "department": "Aseptic Production"},
    {"contract_id": "595207", "contract_number": "595207", "gold_y": 0, "department": "Aseptic Production"},
    
    # Packaging
    {"contract_id": "193",  "contract_number": "193",  "gold_y": 1, "department": "Packaging material"}, 
    {"contract_id": "213",  "contract_number": "213",  "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "190",  "contract_number": "190",  "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "1265", "contract_number": "1265", "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "1245", "contract_number": "1245", "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "1256", "contract_number": "1256", "gold_y": 1, "department": "Packaging material"},
    {"contract_id": "248",  "contract_number": "248",  "gold_y": 0, "department": "Packaging material"},
    {"contract_id": "250",  "contract_number": "250",  "gold_y": 0, "department": "Packaging material"},
    
    # Logistics
    {"contract_id": "192312", "contract_number": "192325", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "541883", "contract_number": "541269", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "582913", "contract_number": "581732", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "569028", "contract_number": "568080", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "470427", "contract_number": "470608", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "530290", "contract_number": "529779", "gold_y": 1, "department": "Logistics"},
    {"contract_id": "533151", "contract_number": "532634", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "550918", "contract_number": "550237", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "525275", "contract_number": "524758", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "335471", "contract_number": "336520", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "607710", "contract_number": "606340", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "536891", "contract_number": "536319", "gold_y": 0, "department": "Logistics"},
    {"contract_id": "599608", "contract_number": "598142", "gold_y": 0, "department": "Logistics"},
]

if MANUAL_LABEL_PATH.exists():
    df_gold_raw = pd.read_csv(MANUAL_LABEL_PATH, dtype={"contract_id": "string", "contract_number": "string"})
    print("Loaded manual labels from:", MANUAL_LABEL_PATH)
else:
    df_gold_raw = pd.DataFrame(gold_label_records)
    print("Loaded manual labels from fallback dictionary.")

if df_gold_raw.empty:
    print("WARNING: No manual gold labels loaded yet. Add labels to the dictionary or Data/manual/gold_labels_manual.csv.")
else:
    display(df_gold_raw.head())
    print("Raw gold label shape:", df_gold_raw.shape)

Loaded manual labels from fallback dictionary.


,contract_id,contract_number,gold_y,department
0,504361,504149,1,Bioprocessing and Excipients
1,613927,612555,1,Bioprocessing and Excipients
2,345318,346312,0,Bioprocessing and Excipients
3,1877,1877,0,Bioprocessing and Excipients
4,4821,4822,1,Global Strategic Outsourcing & Devices & Needles


Raw gold label shape: (33, 4)


## 4. Standardize Gold Label Schema

This section enforces a single canonical schema. A stable schema is important because the gold labels are reused across Stage 1 validation, Stage 2 support/query construction, and thesis diagnostics.

In [71]:
REQUIRED_GOLD_COLUMNS = [
    "contract_id",
    "contract_number",
    "department",
    "gold_y",
    "label_source",
    "label_date",
    "notes",
]

for col in REQUIRED_GOLD_COLUMNS:
    if col not in df_gold_raw.columns:
        df_gold_raw[col] = pd.NA

df_gold = df_gold_raw[REQUIRED_GOLD_COLUMNS].copy()

# Identifier normalization.
df_gold["contract_id"] = df_gold["contract_id"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
df_gold["contract_number"] = df_gold["contract_number"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
df_gold["department"] = df_gold["department"].astype("string").str.strip()

# Label normalization.
df_gold["gold_y"] = pd.to_numeric(df_gold["gold_y"], errors="coerce")

invalid_labels = df_gold[~df_gold["gold_y"].isin([0, 1]) & df_gold["gold_y"].notna()]
if not invalid_labels.empty:
    raise ValueError("Invalid gold_y values detected. Only 0 and 1 are allowed.")

df_gold["gold_y"] = df_gold["gold_y"].astype("Int64")

# Fill metadata defaults.
df_gold["label_source"] = df_gold["label_source"].fillna("category_manager")
df_gold["label_date"] = df_gold["label_date"].fillna(str(date.today()))
df_gold["notes"] = df_gold["notes"].fillna("")

# Remove empty rows.
df_gold = df_gold[df_gold["gold_y"].notna()].copy()


# Automatically look up the matching contract_number from the original database!
id_to_num_map = df_features.set_index(df_features['contract_id'].astype(str))['contract_number'].to_dict()
df_gold['contract_number'] = df_gold['contract_id'].astype(str).map(id_to_num_map)

display(df_gold.head())
print("Standardized gold label shape:", df_gold.shape)

,contract_id,contract_number,department,gold_y,label_source,label_date,notes
0,504361,504149.0,Bioprocessing and Excipients,1,category_manager,2026-04-15,
1,613927,NaN,Bioprocessing and Excipients,1,category_manager,2026-04-15,
2,345318,NaN,Bioprocessing and Excipients,0,category_manager,2026-04-15,
3,1877,NaN,Bioprocessing and Excipients,0,category_manager,2026-04-15,
4,4821,4822.0,Global Strategic Outsourcing & Devices & Needles,1,category_manager,2026-04-15,


Standardized gold label shape: (33, 7)


## 5. Validate Gold Labels Against the Contract Universe

A gold label is only useful if it maps to a contract that exists in the modeling dataset. This validation checks whether labeled contracts can be found by `contract_id` and, where available, by `contract_number`. Any unmatched labels should be investigated before training.

In [72]:
df_features_ids = df_features.copy()

for col in ["contract_id", "contract_number"]:
    if col in df_features_ids.columns:
        df_features_ids[col] = df_features_ids[col].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()

feature_contract_ids = set(df_features_ids["contract_id"].dropna().unique()) if "contract_id" in df_features_ids else set()
feature_contract_numbers = set(df_features_ids["contract_number"].dropna().unique()) if "contract_number" in df_features_ids else set()

df_gold["exists_by_contract_id"] = df_gold["contract_id"].isin(feature_contract_ids)
df_gold["exists_by_contract_number"] = df_gold["contract_number"].isin(feature_contract_numbers)
df_gold["exists_in_features"] = df_gold["exists_by_contract_id"] | df_gold["exists_by_contract_number"]

unmatched = df_gold[~df_gold["exists_in_features"]].copy()

print("Gold labels:", len(df_gold))
print("Matched labels:", int(df_gold["exists_in_features"].sum()))
print("Unmatched labels:", len(unmatched))

if not unmatched.empty:
    display(unmatched)
    unmatched.to_csv(REPORTS_DIR / "unmatched_gold_labels.csv", index=False)
    print("Saved unmatched labels to reports/gold_labels/unmatched_gold_labels.csv")

Gold labels: 33
Matched labels: 21
Unmatched labels: 12


,contract_id,contract_number,department,gold_y,label_source,label_date,notes,exists_by_contract_id,exists_by_contract_number,exists_in_features
1,613927,NaN,Bioprocessing and Excipients,1,category_manager,2026-04-15,,False,False,False
2,345318,NaN,Bioprocessing and Excipients,0,category_manager,2026-04-15,,False,False,False
3,1877,NaN,Bioprocessing and Excipients,0,category_manager,2026-04-15,,False,False,False
8,25777,NaN,Global Strategic Outsourcing & Devices & Needles,0,category_manager,2026-04-15,,False,False,False
9,577893,NaN,Aseptic Production,0,category_manager,2026-04-15,,False,False,False
10,370849,NaN,Aseptic Production,0,category_manager,2026-04-15,,False,False,False
11,595207,NaN,Aseptic Production,0,category_manager,2026-04-15,,False,False,False
25,530290,NaN,Logistics,1,category_manager,2026-04-15,,False,False,False
27,550918,NaN,Logistics,0,category_manager,2026-04-15,,False,False,False
30,607710,NaN,Logistics,0,category_manager,2026-04-15,,False,False,False


Saved unmatched labels to reports/gold_labels/unmatched_gold_labels.csv


## 6. Resolve Duplicate or Conflicting Labels

Gold labels should be unique at the contract level. If the same contract receives both a positive and negative label, the conflict must be resolved manually. Conflicting labels are not automatically averaged or overwritten because this would contaminate the ground truth used for evaluation.

In [73]:
# Prefer contract_id when available; otherwise fall back to contract_number.
df_gold["gold_join_key"] = np.where(
    df_gold["contract_id"].notna() & (df_gold["contract_id"].astype(str) != ""),
    df_gold["contract_id"].astype(str),
    df_gold["contract_number"].astype(str),
)

conflict_check = (
    df_gold.groupby("gold_join_key")
    .agg(
        n_labels=("gold_y", "count"),
        n_unique_labels=("gold_y", "nunique"),
        labels=("gold_y", lambda x: sorted(x.dropna().unique().tolist())),
        departments=("department", lambda x: sorted(x.dropna().unique().tolist())),
    )
    .reset_index()
)

conflicts = conflict_check[conflict_check["n_unique_labels"] > 1].copy()

if not conflicts.empty:
    display(conflicts)
    conflicts.to_csv(REPORTS_DIR / "conflicting_gold_labels.csv", index=False)
    raise ValueError("Conflicting gold labels found. Resolve them before continuing.")

# Keep one label per contract key.
df_gold_clean = (
    df_gold.sort_values(["gold_join_key", "label_date"])
    .drop_duplicates(subset=["gold_join_key"], keep="last")
    .copy()
)

print("Gold labels after duplicate resolution:", len(df_gold_clean))

Gold labels after duplicate resolution: 33


## Random gold labels selection for validation

In [74]:
# --- ADDING DUMMY LABELS TO MASTER LIST ---
np.random.seed(42)
TARGET_K = 12

dummy_rows = []
all_depts = df_features['department'].dropna().unique()

# We check each department and add dummies if needed
for dept in all_depts:
    dept_manual = df_gold_clean[df_gold_clean['department'] == dept]
    existing_ids = set(df_gold_clean['contract_id'])
    
    for label_val in [0, 1]:
        current_count = len(dept_manual[dept_manual['gold_y'] == label_val])
        if current_count < TARGET_K:
            needed = TARGET_K - current_count
            
            # Find candidate contracts in this department not already in the list
            candidates = df_features[
                (df_features['department'] == dept) & 
                (~df_features['contract_id'].astype(str).isin(existing_ids))
            ]['contract_id'].unique()
            
            if len(candidates) > 0:
                sampled = np.random.choice(candidates, size=min(needed, len(candidates)), replace=False)
                for cid in sampled:
                    dummy_rows.append({
                        'contract_id': str(cid),
                        'department': dept,
                        'gold_y': label_val,
                        'label_source': 'DUMMY_FILLER',
                        'label_date': '2026-04-15'
                    })
                    existing_ids.add(str(cid))

# Add them to the master list
if dummy_rows:
    df_gold_clean = pd.concat([df_gold_clean, pd.DataFrame(dummy_rows)], ignore_index=True)
    print(f"✅ Injected {len(dummy_rows)} dummies into the master list.")

print(f"Total labels now in df_gold_clean: {len(df_gold_clean)}")


✅ Injected 271 dummies into the master list.
Total labels now in df_gold_clean: 304


## 7. Gold Label Inventory by Department

This table is the central diagnostic for deciding whether Stage 2 is mathematically feasible. Meta-learning requires departments with both positive and negative examples. A department with only one class cannot support AUROC-based evaluation or balanced support/query episodes.

In [75]:
df_gold_inventory = (
    df_gold_clean.groupby("department", dropna=False)
    .agg(
        gold_total=("gold_y", "count"),
        gold_yes=("gold_y", "sum"),
    )
    .reset_index()
)

df_gold_inventory["gold_no"] = df_gold_inventory["gold_total"] - df_gold_inventory["gold_yes"]
df_gold_inventory["gold_yes_rate"] = df_gold_inventory["gold_yes"] / df_gold_inventory["gold_total"]

df_gold_inventory = df_gold_inventory.sort_values("gold_total", ascending=False)

display(df_gold_inventory)
df_gold_inventory.to_csv(REPORTS_DIR / "gold_label_inventory_by_department.csv", index=False)

,department,gold_total,gold_yes,gold_no,gold_yes_rate
0,"Alliance, Acquisitions & PPM CoE",24,12,12,0.5
15,"Quality, Production Services & Supplies",24,12,12,0.5
2,Bioprocessing & Raw Materials,24,12,12,0.5
3,Bioprocessing and Excipients,24,12,12,0.5
4,Devices & Needles,24,12,12,0.5
6,Drug Product Outsourcing,24,12,12,0.5
7,Drug Substance Outsourcing,24,12,12,0.5
16,Raw Materials & Energy,24,12,12,0.5
11,Logistics,24,12,12,0.5
12,Packaging Material,24,12,12,0.5


## 8. Stage 2 Readiness Check

For k-shot meta-learning, each department needs enough positive and negative labels to form a support set and still leave query examples for evaluation. The rule used here is conservative: a department is valid for k-shot if it has at least `k + 1` positive and `k + 1` negative labels.

In [76]:
K_VALUES = [2, 5, 10]
TARGET_DEPARTMENT = "Logistics"

df_stage2_readiness = df_gold_inventory.copy()

for k in K_VALUES:
    df_stage2_readiness[f"valid_k{k}"] = (
        (df_stage2_readiness["gold_yes"] >= k + 1)
        & (df_stage2_readiness["gold_no"] >= k + 1)
    )

def suggest_role(row):
    if row["department"] == TARGET_DEPARTMENT:
        return "target_candidate"
    if row["gold_total"] == 0:
        return "unlabeled"
    if row["gold_yes"] == 0 or row["gold_no"] == 0:
        return "invalid_one_class"
    if row.get("valid_k2", False):
        return "source_task"
    return "invalid_too_few"

df_stage2_readiness["suggested_role"] = df_stage2_readiness.apply(suggest_role, axis=1)

display(df_stage2_readiness)
df_stage2_readiness.to_csv(REPORTS_DIR / "stage2_gold_label_readiness.csv", index=False)

,department,gold_total,gold_yes,gold_no,gold_yes_rate,valid_k2,valid_k5,valid_k10,suggested_role
0,"Alliance, Acquisitions & PPM CoE",24,12,12,0.5,True,True,True,source_task
15,"Quality, Production Services & Supplies",24,12,12,0.5,True,True,True,source_task
2,Bioprocessing & Raw Materials,24,12,12,0.5,True,True,True,source_task
3,Bioprocessing and Excipients,24,12,12,0.5,True,True,True,source_task
4,Devices & Needles,24,12,12,0.5,True,True,True,source_task
6,Drug Product Outsourcing,24,12,12,0.5,True,True,True,source_task
7,Drug Substance Outsourcing,24,12,12,0.5,True,True,True,source_task
16,Raw Materials & Energy,24,12,12,0.5,True,True,True,source_task
11,Logistics,24,12,12,0.5,True,True,True,target_candidate
12,Packaging Material,24,12,12,0.5,True,True,True,source_task


## 9. Save the Canonical Gold Label Table

The output of this section is the authoritative gold-label file used by downstream notebooks. The file should be version controlled if it does not contain confidential information, or stored securely with a clear versioning procedure if it does.

In [77]:
GOLD_CLEAN_PATH = DATA_PROCESSED / "gold_labels_clean.csv"

df_gold_clean_export = df_gold_clean[
    [
        "contract_id",
        "contract_number",
        "department",
        "gold_y",
        "label_source",
        "label_date",
        "notes",
        "gold_join_key",
    ]
].copy()

df_gold_clean_export.to_csv(GOLD_CLEAN_PATH, index=False)

print("Saved clean gold labels to:", GOLD_CLEAN_PATH)
print("Shape:", df_gold_clean_export.shape)

Saved clean gold labels to: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/processed/gold_labels_clean.csv
Shape: (304, 8)


## 10. Join Gold Labels onto the Feature Dataset

This join produces a modeling-ready dataset with gold labels attached. Snorkel weak labels and gold labels serve different purposes: Snorkel labels provide scalable training signal, while gold labels are used for validation, support/query construction, and final evaluation.

In [78]:
df_features_joined = df_features_ids.copy()

# Remove old gold columns if present to avoid stale labels.
for col in ["gold_y", "gold_department", "label_source", "label_date", "notes"]:
    if col in df_features_joined.columns:
        df_features_joined = df_features_joined.drop(columns=[col])

join_cols = ["contract_id", "contract_number", "department", "gold_y", "label_source", "label_date", "notes"]
df_gold_for_join = df_gold_clean_export[join_cols].copy()

# Primary join by contract_id.
df_features_with_gold = df_features_joined.merge(
    df_gold_for_join.drop(columns=["contract_number"], errors="ignore"),
    on="contract_id",
    how="left",
    suffixes=("", "_gold"),
)

# Preserve department from the gold label as a separate audit column.
if "department_gold" in df_features_with_gold.columns:
    df_features_with_gold = df_features_with_gold.rename(columns={"department_gold": "gold_department"})
else:
    df_features_with_gold["gold_department"] = pd.NA

print("Joined feature table shape:", df_features_with_gold.shape)
print("Gold-labeled rows after join:", int(df_features_with_gold["gold_y"].notna().sum()))
print("Gold-labeled unique contracts after join:", df_features_with_gold.loc[df_features_with_gold["gold_y"].notna(), "contract_id"].nunique())

Joined feature table shape: (9201, 229)
Gold-labeled rows after join: 1269
Gold-labeled unique contracts after join: 292


## Dummy gold labels random selection for pipeline validation

## 11. Save Feature Dataset with Gold Labels

This saved dataset can be used for Stage 1 gold-label validation and Stage 2 support/query construction. If the dataset already contains Snorkel outputs, those columns are preserved.

In [79]:
FEATURES_WITH_GOLD_PATH = DATA_PROCESSED / "contract_with_features_gold_joined.csv"

df_features_with_gold.to_csv(FEATURES_WITH_GOLD_PATH, index=False)

print("Saved feature dataset with gold labels to:", FEATURES_WITH_GOLD_PATH)
print("Shape:", df_features_with_gold.shape)

Saved feature dataset with gold labels to: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/processed/contract_with_features_gold_joined.csv
Shape: (9201, 229)


In [86]:
df_features_with_gold.head()

,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,supplier_display_name,payment_terms,incoterms,contract_currency_code,contract_value,contract_value_dkk,contract_type,nn_contract_type,contract_commodity,team,unit,company_country,days_until_expiry,expiry_range,Preferred_supplier_tag,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,years_to_expiry,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department,moodys_bvd_id,fin_moodys_company_name,fin_closing_year,fin_closing_date,fin_created_at_utc,fin_Status,fin_implied_rating,fin_risk_level,fin_financial_level,fin_output_text,fin_Implied_rating - original,fin_Number_of_months,fin_Net_Salesth_DKK,fin_Operating_revenue_Turnover_th_DKK,fin_Gross_profit_th_DKK,fin_EBIT_th_DKK,fin_Net_income_th_DKK,fin_Interest_paid_th_DKK,fin_EBITDA_th_DKK,fin_Costs_of_goods_soldth_DKK,fin_Total_assets_th_DKK,fin_Non-current_assetsth_DKK,fin_Current_assets_th_DKK,...,fin_flag_strong_liquidity,fin_flag_gearing_high,fin_flag_long_term_gearing_high,fin_flag_short_term_gearing_high,fin_flag_debt_asset_high,fin_flag_debt_asset_very_high,fin_flag_long_term_liab_equity_high,fin_flag_short_term_liab_equity_high,fin_flag_interest_coverage_stress,fin_flag_interest_coverage_weak,fin_flag_low_solvency,fin_flag_very_low_solvency,fin_flag_negative_profit_margin,fin_flag_negative_ebit_margin,fin_flag_profitability_stress,fin_flag_negative_roe,fin_flag_negative_roa,fin_total_stress_flags,fin_flag_multiple_financial_stress_signals,fin_flag_severe_financial_stress,market_has_stock_view,market_flag_high_volume_shock,market_flag_high_market_cap_volatility,market_flag_negative_volume_trend,market_flag_negative_price_trend,market_flag_negative_52w_price_trend,market_flag_high_beta,market_flag_negative_eps,market_flag_stock_price_take_caution_or_worse,market_log_vol_shock_ratio,financial_row_missing_pct,esg_row_missing_pct,news_row_missing_pct,contract_name_lower,contracts_per_supplier,has_environmental_appendix,renegotiation_prob,target_renegotiate,lf_yes_votes,lf_no_votes,lf_abstain_votes,global_lifecycle_yes_votes,global_lifecycle_no_votes,global_financial_yes_votes,global_financial_no_votes,global_esg_yes_votes,global_esg_no_votes,global_news_yes_votes,global_news_no_votes,global_market_yes_votes,global_market_no_votes,global_supplier_macro_yes_votes,global_supplier_macro_no_votes,logistics_specific_yes_votes,logistics_specific_no_votes,gold_department,gold_y,label_source,label_date,notes
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,BIORELIANCE LIMITED INVITROGEN BIOSERVICES,F030 Invoice Date + 30 days,DDP Delivered duty paid,GBP,0.0,0.0,Production,Master Service Agreement,Outsourced GMP Laboratory Analysis,"Quality, Production Services & Supplies",SSIMS,DENMARK,516.0,Over 360 days,0.0,2018,2027,0,2025,2018,2018,9,0,5y_plus,"Distribution, GxP services & Energy","Quality, Production Services & Supplies",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,64.634146,100.0,100.0,bioreliance_master_2018_msa,2,NaN,0.500000,1,0,0,39,0,0,0,0,0,0,0,0,0,0,0,0,0,0,<NA>,<NA>,NaN,NaN,NaN
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,BIORELIANCE LIMITED INVITROGEN BIOSERVICES,F030 Invoice Date + 30 days,DDP Delivered duty paid,GBP,0.0,0.0,Production,Master Service Agreement,Outsourced GMP Laboratory Analysis,"Quality, Production Services & Supplies",SSIMS,DENMARK,516.0,Over 360 days,0.0,2018,2027,0,2025,2018,2019,8,1,5y_plus,"Distribution, GxP services & Energy","Quality, Production Services & Supplies",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0

# Candidate Sampling for Additional Gold Labels

The next cells create candidate lists for category managers. These samples are not gold labels. They are only a structured way to request more human annotations, especially from departments/classes that are currently underrepresented.

## 12. Candidate Sampling Strategy

For each selected department, we sample three groups of contracts:

1. High-probability candidates: likely renegotiation examples.
2. Low-probability candidates: likely non-renegotiation examples.
3. Uncertain candidates: useful for clarifying ambiguous decision boundaries.

This helps collect balanced labels efficiently without pretending that model scores are ground truth.

In [80]:
CANDIDATE_DEPARTMENTS = [
    "Logistics",
    "Packaging Material",
    "Drug Product Outsourcing",
    "Drug Substance Outsourcing",
    "Raw Materials & Energy",
    "Quality, Production Services & Supplies",
]

N_HIGH = 10
N_LOW = 10
N_UNCERTAIN = 10
SCORE_COL = "renegotiation_prob"

if SCORE_COL not in df_features_with_gold.columns:
    raise ValueError(f"Cannot sample candidates because {SCORE_COL} is missing.")

candidate_base = df_features_with_gold.copy()

# One row per contract for review candidates.
candidate_base = (
    candidate_base.sort_values(["contract_id", "observation_year"] if "observation_year" in candidate_base else ["contract_id"])
    .drop_duplicates(subset=["contract_id"], keep="last")
    .copy()
)

# Exclude already labeled contracts.
candidate_base = candidate_base[candidate_base["gold_y"].isna()].copy()

print("Candidate pool unique contracts:", candidate_base["contract_id"].nunique())

Candidate pool unique contracts: 1917


In [81]:
def sample_label_candidates(
    df: pd.DataFrame,
    department: str,
    score_col: str = "renegotiation_prob",
    n_high: int = 10,
    n_low: int = 10,
    n_uncertain: int = 10,
) -> pd.DataFrame:
    dept_df = df[df["department"] == department].copy()
    dept_df = dept_df[dept_df[score_col].notna()].copy()

    if dept_df.empty:
        return pd.DataFrame()

    high = dept_df.sort_values(score_col, ascending=False).head(n_high).copy()
    high["candidate_bucket"] = "likely_yes_high_score"

    low = dept_df.sort_values(score_col, ascending=True).head(n_low).copy()
    low["candidate_bucket"] = "likely_no_low_score"

    uncertain = dept_df.assign(distance_to_half=(dept_df[score_col] - 0.5).abs())
    uncertain = uncertain.sort_values("distance_to_half", ascending=True).head(n_uncertain).copy()
    uncertain["candidate_bucket"] = "uncertain_near_0_5"

    out = pd.concat([high, low, uncertain], ignore_index=True)
    out = out.drop_duplicates(subset=["contract_id"], keep="first")
    out["target_department"] = department

    keep_cols = [
        "target_department",
        "candidate_bucket",
        "contract_id",
        "contract_number",
        "contract_name",
        "supplier_id",
        "supplier_display_name",
        "department",
        score_col,
        "contract_age_years",
        "years_to_expiry_capped",
        "payment_terms",
        "incoterms",
        "contract_value_dkk",
    ]
    keep_cols = [col for col in keep_cols if col in out.columns]

    return out[keep_cols].copy()

In [82]:
candidate_frames = []

for dept in CANDIDATE_DEPARTMENTS:
    df_candidates_dept = sample_label_candidates(
        candidate_base,
        department=dept,
        score_col=SCORE_COL,
        n_high=N_HIGH,
        n_low=N_LOW,
        n_uncertain=N_UNCERTAIN,
    )

    if not df_candidates_dept.empty:
        safe_dept_name = dept.lower().replace(" ", "_").replace("&", "and").replace("/", "_")
        output_path = REPORTS_DIR / f"gold_label_candidates_{safe_dept_name}.csv"
        df_candidates_dept.to_csv(output_path, index=False)
        candidate_frames.append(df_candidates_dept)
        print(f"Saved {len(df_candidates_dept)} candidates for {dept}: {output_path.name}")
    else:
        print(f"No candidates found for {dept}")

if candidate_frames:
    df_all_candidates = pd.concat(candidate_frames, ignore_index=True)
    df_all_candidates.to_csv(REPORTS_DIR / "gold_label_candidates_all_departments.csv", index=False)
    display(df_all_candidates.head(30))
else:
    print("No candidate files created.")

Saved 28 candidates for Logistics: gold_label_candidates_logistics.csv
Saved 30 candidates for Packaging Material: gold_label_candidates_packaging_material.csv
Saved 30 candidates for Drug Product Outsourcing: gold_label_candidates_drug_product_outsourcing.csv
Saved 30 candidates for Drug Substance Outsourcing: gold_label_candidates_drug_substance_outsourcing.csv
Saved 30 candidates for Raw Materials & Energy: gold_label_candidates_raw_materials_and_energy.csv
Saved 30 candidates for Quality, Production Services & Supplies: gold_label_candidates_quality,_production_services_and_supplies.csv


,target_department,candidate_bucket,contract_id,contract_number,contract_name,supplier_id,supplier_display_name,department,renegotiation_prob,contract_age_years,payment_terms,incoterms,contract_value_dkk
0,Logistics,likely_yes_high_score,46854,46854,TFND_Call-off_2014_NNAS_Primary_FFA_Amendment1...,18077,XPO TRANSPORT SOLUTIONS SUD FRANCE DUP#37124 I...,Logistics,1.000000,11,F030 Invoice Date + 30 days,NaN,NaN
1,Logistics,likely_yes_high_score,46860,46860,XPO_Call-off_2017_NNAS_Primary_FFA_Amendment2_...,18077,XPO TRANSPORT SOLUTIONS SUD FRANCE DUP#37124 I...,Logistics,1.000000,8,F030 Invoice Date + 30 days,NaN,0.000000e+00
2,Logistics,likely_yes_high_score,46847,46847,TFND_Master_2010_NNAS_Primary_FFA,18077,XPO TRANSPORT SOLUTIONS SUD FRANCE DUP#37124 I...,Logistics,1.000000,15,F030 Invoice Date + 30 days,NaN,NaN
3,Logistics,likely_yes_high_score,46134,46135,Leman_Call-off_2015_NNAS_Primary_FFA_Amendment...,8880,LEMAN A/S,Logistics,1.000000,10,F030 Invoice Date + 30 days,NaN,0.000000e+00
4,Logistics,likely_yes_high_score,46139,46140,Leman_Call-off_2017_NNAS_Primary_FFA_Amendment...,8880,LEMAN A/S,Logistics,1.000000,8,F030 Invoice Date + 30 days,NaN,0.000000e+00
5,Logistics,likely_yes_high_score,47409,47410,Neptun_Call-off_2017_NNAS_Primary_FFA_Amendmen...,1708,NEPTUN TRANSPORT A/S,Logistics,1.000000,13,F030 Invoice Date + 30 days,NaN,NaN
6,Logistics,likely_yes_high_score,46918,46918,BHS_Call-off_2017_NNAS_Primary_FFA_Amendment1_...,2820,BHS LOGISTICS A/S,Logistics,1.000000,8,F030 Invoice Date + 30 days,NaN,0.000000e+00
7,Logistics,likely_yes_high_score,1138,1138,DHL GLOBAL FORWARDING_MASTER_2017_NNAS_Primary...,35965,DHL Global Forwarding,Logistics,0.999051,8,F060 Invoice Date + 60 days,NaN,2.200000e+06
8,Logistics,likely_yes_high_score,256511,257767,Kuehne + Nagel A/S_CDA_2022,50022,KUEHNE + NAGEL A/S,Logistics,0.451515,3,F030 Invoice Date + 30 days,NaN,0.000000e+00
9,Logistics,likely_yes_high_score,467944,468104,Kuehne + Nagel_Call-Off_no. 1_RDC_Capacity & m...,272874,KUEHNE NAGEL S PTE LTD,Logistics,0.342602,1,F060 Invoice Date + 60 days,NaN,1.789150e+05


## 13. Final Summary

The final summary connects the label inventory to the thesis methodology. If the gold-label distribution is balanced across several departments, Stage 2 meta-learning becomes more defensible. If labels remain concentrated in one department or one class, the thesis should frame Stage 2 as constrained few-shot adaptation rather than fully general meta-learning.

In [83]:
total_gold = len(df_gold_clean_export)
total_yes = int(df_gold_clean_export["gold_y"].sum()) if total_gold else 0
total_no = int(total_gold - total_yes)

print("--- GOLD LABEL SUMMARY ---")
print("Total gold labels:", total_gold)
print("Gold yes:", total_yes)
print("Gold no:", total_no)
print("Departments with gold labels:", df_gold_clean_export["department"].nunique() if total_gold else 0)

for k in K_VALUES:
    valid_sources = df_stage2_readiness[
        (df_stage2_readiness[f"valid_k{k}"])
        & (df_stage2_readiness["department"] != TARGET_DEPARTMENT)
    ]
    print(f"Valid source departments for k={k}:", len(valid_sources))

logistics_row = df_stage2_readiness[df_stage2_readiness["department"] == TARGET_DEPARTMENT]
if not logistics_row.empty:
    display(logistics_row)
else:
    print("No Logistics gold labels found yet.")

--- GOLD LABEL SUMMARY ---
Total gold labels: 304
Gold yes: 137
Gold no: 167
Departments with gold labels: 19
Valid source departments for k=2: 10
Valid source departments for k=5: 10
Valid source departments for k=10: 9


,department,gold_total,gold_yes,gold_no,gold_yes_rate,valid_k2,valid_k5,valid_k10,suggested_role
11,Logistics,24,12,12,0.5,True,True,True,target_candidate
